# Load the Data

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
# Load the raw datasets
try:
    df_posts = pd.read_csv('/content/raw_kols_posts.csv')
    df_comments = pd.read_csv('/content/raw_comments_data.csv')

    print("Data successfully loaded!")
    print(f"Posts dataset shape: {df_posts.shape}")
    print(f"Comments dataset shape: {df_comments.shape}")

except FileNotFoundError as e:
    print(f"Error: {e}.")

Data successfully loaded!
Posts dataset shape: (150, 1188)
Comments dataset shape: (300, 23)


# Data cleaning

In [3]:
# Clean the Posts Dataset
# Define only the relevant columns needed for the DaaS MVP
posts_cols_to_keep = [
    'id', 'ownerUsername', 'type', 'shortCode',
    'caption', 'commentsCount', 'likesCount', 'timestamp', 'url'
]

# Filter existing columns gracefully to avoid KeyError
existing_posts_cols = [col for col in posts_cols_to_keep if col in df_posts.columns]
df_posts_clean = df_posts[existing_posts_cols].copy()

# Rename columns to standard snake_case
df_posts_clean.rename(columns={
    'ownerUsername': 'kol_username',
    'shortCode': 'post_short_code',
    'commentsCount': 'comments_count',
    'likesCount': 'likes_count'
}, inplace=True)

# Convert timestamp to a proper Datetime object
if 'timestamp' in df_posts_clean.columns:
    df_posts_clean['timestamp'] = pd.to_datetime(df_posts_clean['timestamp'])

print("Posts data cleaned successfully.")
print(f"Remaining columns: {df_posts_clean.columns.tolist()}")

Posts data cleaned successfully.
Remaining columns: ['id', 'kol_username', 'type', 'post_short_code', 'caption', 'comments_count', 'likes_count', 'timestamp', 'url']


In [4]:
# Clean Comments and Merge Data
# Standardize the foreign key (post_short_code) in the comments dataset
if 'postUrl' in df_comments.columns:
    # Extract the short code from Instagram URL (e.g., .../p/SHORTCODE/)
    df_comments['post_short_code'] = df_comments['postUrl'].apply(
        lambda x: str(x).split('/p/')[1].split('/')[0] if pd.notnull(x) and '/p/' in str(x)
        else (str(x).split('/reel/')[1].split('/')[0] if pd.notnull(x) and '/reel/' in str(x) else None)
    )
elif 'postShortCode' in df_comments.columns:
    df_comments['post_short_code'] = df_comments['postShortCode']

# Select relevant comment columns
comments_cols_to_keep = ['id', 'text', 'ownerUsername', 'timestamp', 'likesCount', 'post_short_code']
existing_comment_cols = [col for col in comments_cols_to_keep if col in df_comments.columns]
df_comments_clean = df_comments[existing_comment_cols].copy()

# Rename columns to prevent overlap with posts data
df_comments_clean.rename(columns={
    'id': 'comment_id',
    'text': 'comment_text',
    'ownerUsername': 'commenter_username',
    'timestamp': 'comment_timestamp',
    'likesCount': 'comment_likes_count'
}, inplace=True)

# Perform an INNER JOIN using Pandas (Merge)
df_master = pd.merge(
    df_comments_clean,
    df_posts_clean[['post_short_code', 'kol_username']],
    on='post_short_code',
    how='inner'
)

# Drop any rows where the comment text is empty (NaN)
df_master.dropna(subset=['comment_text'], inplace=True)

print("Data merge successful!")
print(f"Master dataset shape: {df_master.shape}")

Data merge successful!
Master dataset shape: (295, 7)


In [5]:
# Export the master dataset
df_master.to_csv('master_dataset_clean.csv', index=False)
print("Master dataset saved as 'master_dataset_clean.csv'.")

# Display a preview of the final dataset
display(df_master[['kol_username', 'commenter_username', 'comment_text']].head())

Master dataset saved as 'master_dataset_clean.csv'.


,kol_username,commenter_username,comment_text
0,henjiwong,richa_rd2782,@ifwatlh @reynalkristopayan_
1,henjiwong,tenangaza88,Aslinya pasti gak kek gini lah
2,henjiwong,yuliani.lo,A
3,henjiwong,baksojonomukti,Juicy🔥🔥
4,henjiwong,foodsreviewsid,ampunnn seenakk itu :)


# RULE-BASED BOT & SENTIMENT ENGINE

In [6]:
# Load the clean master dataset
try:
    df = pd.read_csv('master_dataset_clean.csv')
    print(f"Dataset loaded successfully. Shape: {df.shape}")

except FileNotFoundError:
    print("Error: 'master_dataset_clean.csv' not found.")

# Ensure comment_text is treated as string and handle NaNs
df['comment_text'] = df['comment_text'].fillna('').astype(str)

Dataset loaded successfully. Shape: (295, 7)


In [7]:
# Bot detection logic rule based
def detect_bot_spam(text):
    text_lower = text.lower()

    # Rule 1: Promotional / Spam Keywords
    spam_keywords = ['cek ig', 'peninggi', 'langsing', 'followers', 'klik link', 'pinjaman', 'slot', 'gacor', 'jual akun', 'jual followers', 'tambah followers']

    for keyword in spam_keywords:
        # \b ensures we match exact whole words/phrases, avoiding substring traps
        if re.search(r'\b' + re.escape(keyword) + r'\b', text_lower):
            return True, "Promotional Spam"

    # Rule 2: Over-tagging
    mentions = re.findall(r'@\w+', text)
    if len(mentions) >= 3:
        return True, "Over-tagging Spam"

    # Rule 3: Low Effort / Gibberish / Only Emojis
    text_clean = text.strip()
    if len(text_clean) <= 2:
        return True, "Low Effort / Emoji Spam"

    # If it passes all rules, it's likely a real human engagement
    return False, "Human"


In [8]:
# Apply bot detection logic to the dataset
df['is_bot_suspicion'], df['bot_reason'] = zip(*df['comment_text'].apply(detect_bot_spam))

In [9]:
# Sentiment analysis logic rule based
def analyze_indonesian_fb_sentiment(text, is_bot):
    # If it's a bot, we don't care about the sentiment (exclude from brand analysis)
    if is_bot:
        return "Excluded (Bot)"

    text_lower = text.lower()

    # Lexicon for Indonesian F&B context
    positive_words = ['enak', 'mantap', 'lezat', 'juara', 'ngiler', 'gila', 'wow', 'sedap', 'suka', 'best', 'favorit', 'kangen', 'mau', 'pengen', 'nyam']
    negative_words = ['mahal', 'basi', 'kecewa', 'zonk', 'kurang', 'amis', 'keras', 'asin', 'bau', 'kotor', 'buruk', 'gaksuka', 'gak enak']

    # Simple scoring system
    score = 0
    for word in positive_words:
        if re.search(r'\b' + word + r'\b', text_lower): # \b ensures exact word match
            score += 1

    for word in negative_words:
        if re.search(r'\b' + word + r'\b', text_lower):
            score -= 1

    # Classify based on score
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        # Fallback to Neutral if no strong keywords are detected
        return "Neutral"

In [10]:
# Apply sentiment logic
df['sentiment'] = df.apply(lambda row: analyze_indonesian_fb_sentiment(row['comment_text'], row['is_bot_suspicion']), axis=1)

In [11]:
# Analytical summary
total_comments = len(df)
total_bots = df['is_bot_suspicion'].sum()
bot_percentage = (total_bots / total_comments) * 100

print(f"Total Comments Analyzed: {total_comments}")
print(f"Suspicious Bots Detected: {total_bots} ({bot_percentage:.2f}%)")
print("\nSentiment Distribution (Real Humans):")
print(df[df['is_bot_suspicion'] == False]['sentiment'].value_counts())

# Export the final modeled data
output_filename = 'final_daas_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"\nPhase 3 complete! Modeled dataset exported as '{output_filename}'.")

# Preview the interesting cases (Bots / Sentiments)
print("\nPreview of detected bots/spam:")
display(df[df['is_bot_suspicion'] == True][['kol_username', 'commenter_username', 'comment_text', 'bot_reason']].head(5))

Total Comments Analyzed: 295
Suspicious Bots Detected: 14 (4.75%)

Sentiment Distribution (Real Humans):
sentiment
Neutral     238
Positive     37
Negative      6
Name: count, dtype: int64

Phase 3 complete! Modeled dataset exported as 'final_daas_dataset.csv'.

Preview of detected bots/spam:


,kol_username,commenter_username,comment_text,bot_reason
2,henjiwong,yuliani.lo,A,Low Effort / Emoji Spam
8,henjiwong,danielsuandikurniawidjaja,B,Low Effort / Emoji Spam
15,sibungbung,daily.moly,😍😍,Low Effort / Emoji Spam
55,anakjajan,marselasusanto,😍😍,Low Effort / Emoji Spam
88,henjiwong,eka26347,V,Low Effort / Emoji Spam
